# AutoCache — Entraînement du modèle « 4 coins de plaque » (YOLOv8-pose → ONNX)

Ce notebook prend votre dataset annoté (export Roboflow YOLOv8 Keypoints) et produit :
1. `best.pt` — le modèle entraîné (pour ré-entraîner plus tard) ;
2. **`plate-keypoints.onnx`** — le fichier à intégrer dans AutoCache (navigateur).

**Vous n'avez besoin d'AUCUNE compétence technique.** Cliquez sur ▶️ à gauche de chaque
cellule, de haut en bas.

**Avant de commencer :** menu `Exécution → Modifier le type d'exécution → GPU T4`
(l'entraînement est ~20× plus rapide sur GPU, et c'est gratuit).

## 1. Installer les dépendances (~1 min)

In [ ]:
!pip install -q ultralytics onnx onnxslim
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.is_available())

## 2. Charger votre dataset

Deux façons — choisissez-en une :

**A) Depuis Roboflow directement** (le plus simple) : sur votre version Roboflow,
`Download Dataset → Format: YOLOv8 → Keypoints → show download code`, et collez
le bout de code `roboflow` proposé à la place de la cellule ci-dessous.

**B) Depuis un zip** : glissez votre export Roboflow (zip) dans le panneau Fichiers
de Colab (icône dossier à gauche), puis exécutez la cellule.

In [ ]:
# --- Option B : décompresser un zip Roboflow déposé dans Colab ---
import glob, zipfile, os, yaml
zips = glob.glob('/content/*.zip')
assert zips, "Aucun .zip trouvé. Déposez votre export Roboflow dans le panneau Fichiers, ou utilisez l'option A."
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('/content/dataset')
# Localise le data.yaml de l'export
yamls = glob.glob('/content/dataset/**/data.yaml', recursive=True)
assert yamls, 'data.yaml introuvable dans le zip.'
DATA_YAML = yamls[0]
print('Dataset prêt :', DATA_YAML)
print(yaml.safe_load(open(DATA_YAML)))

## 3. Vérifier le format des keypoints (garde-fou)

Le rendu du cache dépend de l'ordre strict des 4 coins **tl, tr, br, bl**. Cette
cellule vérifie que l'export a bien `kpt_shape: [4, 3]` avant d'entraîner.

In [ ]:
cfg = yaml.safe_load(open(DATA_YAML))
ks = cfg.get('kpt_shape')
assert ks == [4, 3], f"kpt_shape attendu [4, 3], obtenu {ks}. Vérifiez le squelette Roboflow (4 keypoints tl,tr,br,bl)."
print('OK — 4 keypoints par plaque. flip_idx =', cfg.get('flip_idx'))

## 4. Entraîner (~15–40 min sur GPU T4 selon le nombre de photos)

`yolov8n-pose` = le plus léger (~6 Mo, idéal navigateur). Pour un premier modèle,
gardez-le. Si plus tard vous voulez plus de précision et que le poids ne vous gêne
pas, passez à `yolov8s-pose`.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n-pose.pt')
model.train(
    data=DATA_YAML,
    epochs=120,
    imgsz=640,
    batch=16,
    patience=30,           # arrêt anticipé si plus de progrès
    name='plate-keypoints',
)

## 5. Vérifier la qualité

Regardez `metrics/mAP50` : au-dessus de **~0.7**, le modèle est utilisable. Les
courbes et exemples de prédiction s'affichent ci-dessous.

In [ ]:
from IPython.display import Image, display
import glob
run = sorted(glob.glob('runs/pose/plate-keypoints*'))[-1]
print('Dossier run :', run)
for img in ['results.png', 'val_batch0_pred.jpg']:
    p = f'{run}/{img}'
    if glob.glob(p):
        display(Image(p))

## 6. Exporter en ONNX pour le navigateur

`opset=12` + taille fixe 640 = compatible onnxruntime-web (le moteur qu'AutoCache
charge déjà pour le détourage).

In [ ]:
best = f'{run}/weights/best.pt'
m = YOLO(best)
onnx_path = m.export(format='onnx', opset=12, imgsz=640, simplify=True, dynamic=False)
print('ONNX exporté :', onnx_path)

## 7. Télécharger les fichiers

Récupérez **`plate-keypoints.onnx`** (à me transmettre pour l'intégration) et
gardez `best.pt` de côté pour ré-entraîner plus tard sur un dataset élargi.

In [ ]:
import shutil
from google.colab import files
shutil.copy(onnx_path, '/content/plate-keypoints.onnx')
shutil.copy(best,      '/content/best.pt')
files.download('/content/plate-keypoints.onnx')
files.download('/content/best.pt')